<a href="https://colab.research.google.com/github/BreakTheAlgorithm/MLforALL/blob/main/book_ch12_dimensionality_reduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<div style='background:linear-gradient(135deg,#1B3A6B,#2D5AA0);padding:30px;border-radius:12px;color:white;font-family:Arial'>
<h1 style='margin:0;font-size:28px'>📉 Chapter 12 — Dimensionality Reduction</h1>
<p style='margin:8px 0 0'>Book: Machine Learning — From Zero to Engineer | Est. Time: 60 mins | Level: Intermediate</p>
</div>

## 📋 Learning Objectives

By the end of this notebook, you will be able to:
- Explain the curse of dimensionality
- Apply PCA and interpret explained variance ratio
- Use t-SNE for 2D visualisation of high-dimensional data
- Choose between PCA (preprocessing) and t-SNE (visualisation only)
- Use PCA as a preprocessing step to improve model speed and accuracy

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Setup & Imports
# ─────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition   import PCA
from sklearn.manifold        import TSNE
from sklearn.preprocessing   import StandardScaler
from sklearn.datasets        import load_digits, make_classification
from sklearn.ensemble        import RandomForestClassifier
from sklearn.model_selection import cross_val_score

%matplotlib inline
np.random.seed(42)

# MNIST subset — handwritten digits (1797 samples, 64 features)
digits = load_digits()
X_dig, y_dig = digits.data, digits.target
print(f'Digits dataset: {X_dig.shape}, classes: {np.unique(y_dig)}')

## 📖 Section A — Principal Component Analysis (PCA)

PCA finds directions (components) of maximum variance in the data:

- **PC1**: direction of highest variance
- **PC2**: direction of second highest variance, orthogonal to PC1
- ...

The `explained_variance_ratio_` tells you how much of the total variance each component captures.

```python
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_reduced = pca.fit_transform(X_scaled)
print(pca.explained_variance_ratio_)  # e.g. [0.45, 0.22]
print('Total variance retained:', sum(pca.explained_variance_ratio_))
```

> 💡 **Key Rule:** Always StandardScale before PCA. PCA is based on variance — a feature with large values will dominate the components. After scaling, each feature has equal opportunity to contribute.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Demo: PCA on Digits Dataset
# ─────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_dig)

pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)

plt.figure(figsize=(9, 4))
plt.plot(range(1, len(cumvar)+1), cumvar, 'b-o', markersize=3)
plt.axhline(0.95, color='red', linestyle='--', label='95% variance')
plt.xlabel('Number of Components')
plt.ylabel('Cumulative Explained Variance')
plt.title('PCA — Explained Variance Curve')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

n_95 = np.argmax(cumvar >= 0.95) + 1
print(f'Components needed for 95% variance: {n_95} (out of {X_dig.shape[1]})')
print(f'Dimensionality reduction: {X_dig.shape[1]} → {n_95} ({100*(1-n_95/X_dig.shape[1]):.0f}% smaller)')

## 📖 Section B — t-SNE for Visualisation

t-SNE (t-distributed Stochastic Neighbour Embedding) is a non-linear method for 2D/3D visualisation:

- Preserves **local** structure — nearby points in high-D stay nearby in 2D
- NOT suitable for preprocessing before ML models (not deterministic, not invertible)
- Good for visual exploration only

```python
from sklearn.manifold import TSNE
X_tsne = TSNE(n_components=2, random_state=42, perplexity=30).fit_transform(X_scaled)
```

> 💡 **Key Rule:** Use PCA first to reduce to ~50 dimensions, then apply t-SNE — this dramatically speeds up t-SNE without losing meaningful structure.

---
## 🟢 Exercise 12.1 — PCA 2D Visualisation *(Level 1 · Est. 10 min)*

> Reduce the digits dataset to 2 components with PCA. Scatter plot PC1 vs PC2, coloured by digit class. How well are the 10 digit classes separated?

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 12.1: PCA 2D Visualisation
# ─────────────────────────────────────────────────────────────

pca2 = PCA(n_components=2, random_state=42)
# YOUR CODE HERE — fit_transform X_scaled

plt.figure(figsize=(9, 6))
# YOUR CODE HERE — scatter plot coloured by digit class

plt.title('PCA — 2D Projection of Digits Dataset')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

print(f'Variance retained by PC1+PC2: {sum(pca2.explained_variance_ratio_):.2%}')

In [ ]:
# @title ✅ Solution — Exercise 12.1 (click to expand)

pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))
scatter = plt.scatter(X_pca2[:, 0], X_pca2[:, 1], c=y_dig,
                      cmap='tab10', alpha=0.6, s=15)
plt.colorbar(scatter, label='Digit Class')
plt.title('PCA — 2D Projection of Digits Dataset')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.show()

print(f'Variance retained by PC1+PC2: {sum(pca2.explained_variance_ratio_):.2%}')
print('✅ Notice clusters form but overlap — 2 components retain only ~28% variance. Some classes hard to separate here.')

---
## 🟡 Exercise 12.2 — PCA as Preprocessing Step *(Level 2 · Est. 15 min)*

> Compare Random Forest accuracy with and without PCA preprocessing. Use k=95% components. Time the fit for each. Report accuracy, training time, and components used.

In [ ]:
# ─────────────────────────────────────────────────────────────
# Exercise 12.2: PCA as Preprocessing
# ─────────────────────────────────────────────────────────────
import time
from sklearn.pipeline import Pipeline

results = []

# Model 1: No PCA
# YOUR CODE HERE

# Model 2: With PCA (95% variance)
# YOUR CODE HERE

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print('\n✅ PCA preprocessing analysis complete!')

In [ ]:
# @title ✅ Solution — Exercise 12.2 (click to expand)
import time
from sklearn.pipeline        import Pipeline
from sklearn.model_selection import StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
results = []

# Model 1: No PCA
t0 = time.time()
pipe1 = Pipeline([('scaler', StandardScaler()), ('rf', RandomForestClassifier(n_estimators=50, random_state=42))])
acc1 = cross_val_score(pipe1, X_dig, y_dig, cv=cv, scoring='accuracy').mean()
results.append({'Pipeline': 'No PCA', 'Accuracy': round(acc1, 4), 'Time(s)': round(time.time()-t0, 2), 'Features': X_dig.shape[1]})

# Model 2: With PCA 95%
t0 = time.time()
pca95 = PCA(n_components=0.95, random_state=42)
pipe2 = Pipeline([('scaler', StandardScaler()), ('pca', pca95), ('rf', RandomForestClassifier(n_estimators=50, random_state=42))])
acc2 = cross_val_score(pipe2, X_dig, y_dig, cv=cv, scoring='accuracy').mean()
# Fit once to check components
from sklearn.model_selection import train_test_split
X_tr, X_te, y_tr, y_te = train_test_split(X_dig, y_dig, random_state=42)
pca_temp = Pipeline([('scaler', StandardScaler()), ('pca', PCA(n_components=0.95, random_state=42))])
pca_temp.fit(X_tr)
n_components = pca_temp.named_steps['pca'].n_components_
results.append({'Pipeline': 'PCA 95%', 'Accuracy': round(acc2, 4), 'Time(s)': round(time.time()-t0, 2), 'Features': n_components})

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))
print(f'\nDimensionality: {X_dig.shape[1]} → {n_components} features')
print('✅ PCA typically maintains accuracy while reducing features and sometimes training time.')

---
## 🔴 Exercise 12.3 — t-SNE Visualisation *(Level 3 · Est. 20 min)*

> Apply t-SNE to digits (after PCA to 30 dims first). Plot 2D t-SNE coloured by class. Compare visually to PCA 2D. Discuss which shows better cluster separation.

In [ ]:
# @title ✅ Solution — Exercise 12.3 (click to expand)

# Step 1: PCA to 30 dims (speeds up t-SNE)
pca30 = PCA(n_components=30, random_state=42)
X_pca30 = pca30.fit_transform(X_scaled)
print(f'PCA 30 components retain: {sum(pca30.explained_variance_ratio_):.2%} variance')

# Step 2: t-SNE to 2D
print('Running t-SNE (may take ~30s)...')
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_pca30)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, data, title in [
    (axes[0], X_pca2, 'PCA 2D'),
    (axes[1], X_tsne, 't-SNE 2D')
]:
    sc = ax.scatter(data[:, 0], data[:, 1], c=y_dig, cmap='tab10', alpha=0.6, s=15)
    plt.colorbar(sc, ax=ax)
    ax.set_title(title)

plt.suptitle('PCA vs t-SNE: 2D Projections of Digits', fontsize=13)
plt.tight_layout()
plt.show()
print('\n✅ t-SNE shows MUCH better class separation. PCA is linear — struggles with non-linear structure.')
print('Reminder: t-SNE is ONLY for visualisation — do NOT use it as preprocessing for ML models.')

---
## 🎤 Interview Q&A

<details>
<summary><strong>Q1: What is the curse of dimensionality?</strong></summary>

**Answer:** As the number of features grows, the volume of the feature space increases exponentially. Data points become increasingly sparse — the nearest neighbour is barely closer than the farthest neighbour. Models need exponentially more data to learn effectively. Distance-based algorithms (KNN, K-Means, SVMs with RBF kernels) suffer most. Feature selection and dimensionality reduction (PCA) combat this.
</details>

<details>
<summary><strong>Q2: What does PCA's explained variance ratio tell you?</strong></summary>

**Answer:** It tells you what fraction of the total variance in the original data each principal component explains. For example, [0.45, 0.22, 0.12] means PC1 explains 45%, PC2 22%, PC3 12% — cumulative 79%. Choosing components that explain ≥95% variance is a common threshold. The key insight: if 95% of variance can be captured in 20% of the original dimensions, the remaining 80% of features mostly contain noise or redundancy.
</details>

<details>
<summary><strong>Q3: Why can't you use t-SNE as a preprocessing step for model training?</strong></summary>

**Answer:** (1) t-SNE is non-deterministic — results change each run. (2) It's not invertible — you cannot transform new test data with the same mapping. (3) It only preserves local structure, not global distances. (4) It's computationally expensive O(n²). Use t-SNE ONLY for visualisation. Use PCA (which is invertible and deterministic) as a preprocessing step. Apply the same PCA transform (fitted on training data) to test data.
</details>

---

<div style='background:#EDF7F0;border-left:6px solid #2E7D50;padding:20px;border-radius:8px;margin-top:20px'>
<h3 style='color:#2E7D50'>✅ Chapter 12 Complete!</h3>
<ul>
<li>PCA: explained variance, components selection</li>
<li>Using PCA as preprocessing to speed up models</li>
<li>t-SNE for visual exploration of high-dimensional data</li>
<li>PCA vs t-SNE — when to use each</li>
</ul>
<p><strong>Next:</strong> Chapter 13 — Natural Language Processing Basics</p>
</div>